# 04 — GraphSAGE controlled relation experiments

The only intended difference between the GraphSAGE experiments is `edge_index`:

1. `card1 + addr1` (6.39M edges in original run)
2. `card1 + DeviceInfo` (~0.97M)
3. deduplicated union (~7.27M)

Every experiment uses the shared 397/37 input schema, one-layer **mean GraphSAGE**, 128 hidden dimensions, weighted BCE and validation Average Precision selection. Our PyG batch contains both seed nodes and historical feature-only context nodes; loss is computed only on seed labels.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository:", ROOT)

In [ ]:
import torch
from fraud_graph.data import load_processed
from fraud_graph.paths import PROCESSED_DIR
arr, meta = load_processed()
for name, file in [("address", "edge_index_v1.pt"),
                   ("device", "edge_index_device_v1.pt"),
                   ("combined", "edge_index_combined_v1.pt")]:
    path = PROCESSED_DIR / file
    if path.exists():
        edge_index = torch.load(path, weights_only=True, map_location="cpu")
        print(name, edge_index.shape)
    else:
        print("Not yet built:", name)

## Why sampler is necessary

`NeighborLoader(num_neighbors=[-1])` collects all immediate *incoming* predecessors (each base relation has maximum indegree 20). Seed node labels are supervised; context labels are never used by the GNN. `pyg-lib` or `torch-sparse` must match your exact PyTorch build. Read the full sampler and training code in `src/fraud_graph/data.py` and `src/fraud_graph/training.py`.

In [ ]:
import subprocess
RUN_TRAINING = False
if RUN_TRAINING:
    for relation in ("address", "device", "combined"):
        print("Training graph:", relation, flush=True)
        subprocess.run(["uv", "run", "python", "scripts/train_gnn.py",
                        "--graph", relation, "--seed", "42"], cwd=ROOT, check=True)
else:
    print("Set RUN_TRAINING=True to repeat all three GNN experiments.")